# VSHORAD - Section 6.2: Swin Classifier Evaluation

**Goal**: Generate all evaluation materials for the Swin classifiers.

| Tier | Model | img_size | Parameters |
|------|-------|----------|------------|
| **Strategic** | Swin-Base | 384 | ~88M |
| **Tactical** | Swin-Small | 224 | ~50M |

# 1. Setup

In [ ]:
!pip install -q timm==1.0.11 grad-cam==1.5.4 albumentations==1.4.18
!pip install -q pandas matplotlib seaborn scikit-learn fvcore
print(" Installation complete")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, zipfile, shutil, random
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, top_k_accuracy_score

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

try:
  from fvcore.nn import FlopCountAnalysis
  FVCORE_AVAILABLE = True
except:
  FVCORE_AVAILABLE = False

plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device: {DEVICE}")
if torch.cuda.is_available():
  print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Path config

BASE_PATH = Path("/content/drive/MyDrive/Inżynierka")

# Dataset
DATASETS_PATH = BASE_PATH / "Datasety Finalne/Datasety_Zbalansowane"
CLASSIFICATION_ZIP = DATASETS_PATH / "Skydetect_Balanced_Swin_Dataset.zip"
DATASET_EXTRACT_PATH = Path("/content/datasets/classification")

# Swin models
SWIN_PATHS = {
  'strategic': {
    'checkpoint': BASE_PATH / 'Strategic_components/checkpoints/swin/strategic_swin_base_384/best.pth',
    'training_log': BASE_PATH / 'Strategic_components/checkpoints/swin/strategic_swin_base_384/training_log.json',
    'model_name': 'swin_base_patch4_window12_384',
    'img_size': 384,
    'tier_name': 'Strategic',
    'display_name': 'Swin-Base@384'
  },
  'tactical': {
    'checkpoint': BASE_PATH / 'Medium_components/checkpoints/swin/medium_swin_small_224/best.pth',
    'training_log': BASE_PATH / 'Medium_components/checkpoints/swin/medium_swin_small_224/training_log.json',
    'model_name': 'swin_small_patch4_window7_224',
    'img_size': 224,
    'tier_name': 'Tactical',
    'display_name': 'Swin-Small@224'
  }
}

# Output directories
REPORTS_BASE = BASE_PATH / "Raporty"
OUTPUT_DIRS = {
  'swin': REPORTS_BASE / "Swin",
  'tables': REPORTS_BASE / "Swin/tables",
  'confusion_matrices': REPORTS_BASE / "Swin/confusion_matrices",
  'training_curves': REPORTS_BASE / "Swin/training_curves",
  'gradcam_strategic': REPORTS_BASE / "Swin/gradcam/strategic",
  'gradcam_tactical': REPORTS_BASE / "Swin/gradcam/tactical",
  'comparisons': REPORTS_BASE / "Swin/comparisons",
  'raport': REPORTS_BASE / "Swin/raport_claude",
}

print(" Creating folder structure:")
for name, path in OUTPUT_DIRS.items():
  path.mkdir(parents=True, exist_ok=True)
  print(f"  {name}: {path}")

for tier in ['strategic', 'tactical']:
  for cat in ['confident', 'uncertain', 'incorrect']:
    (OUTPUT_DIRS[f'gradcam_{tier}'] / cat).mkdir(parents=True, exist_ok=True)

REPORT_DIR = OUTPUT_DIRS['raport']
(REPORT_DIR / 'plots').mkdir(exist_ok=True)

In [ ]:
# Classes loaded dynamically from dataset or checkpoint

# Placeholder - overwritten after loading dataset/checkpoint
CLASS_NAMES = None
NUM_CLASSES = None

def build_meta_categories(class_names):
  """Buduje meta-kategorie na podstawie listy classes."""
  meta = {
    'Fighter': [c for c in class_names if c.startswith('Fighter_')],
    'Helicopter': [c for c in class_names if c.startswith('Helicopter_')],
    'Bomber': [c for c in class_names if c.startswith('Bomber_')],
    'Transport': [c for c in class_names if c.startswith('Transport_')],
    'Special': [c for c in class_names if c.startswith('Special_')],
    'Decoys': [c for c in class_names if c.startswith('Decoys_')],
  }
  # Dodaj pojedyncze classesy
  for c in class_names:
    if not any(c in v for v in meta.values()):
      meta[c] = [c]
  # Remove empty categories
  return {k: v for k, v in meta.items() if v}

print(" Classes will be loaded from dataset/checkpoint")

In [ ]:
# Extract dataset and get class names

print(" Checking dataset...")
DATASET_PATH = None

if DATASET_EXTRACT_PATH.exists() and any(DATASET_EXTRACT_PATH.iterdir()):
  print("  Dataset already extracted")
else:
  if CLASSIFICATION_ZIP.exists():
    print(f"  Extracting...")
    DATASET_EXTRACT_PATH.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CLASSIFICATION_ZIP, 'r') as zf:
      zf.extractall(DATASET_EXTRACT_PATH)
    print("  Extracted")
  else:
    print(f"  Brak ZIP: {CLASSIFICATION_ZIP}")

# Find main folder
dataset_contents = list(DATASET_EXTRACT_PATH.iterdir())
if len(dataset_contents) == 1 and dataset_contents[0].is_dir():
  DATASET_PATH = dataset_contents[0]
else:
  DATASET_PATH = DATASET_EXTRACT_PATH

print(f"  Dataset: {DATASET_PATH}")

# POBIERZ KLASY Z DATASETU (sortowane alfabetycznie jak ImageFolder)
val_path = DATASET_PATH / 'val'
if val_path.exists():
  CLASS_NAMES = sorted([d.name for d in val_path.iterdir() if d.is_dir()])
  NUM_CLASSES = len(CLASS_NAMES)
  META_CATEGORIES = build_meta_categories(CLASS_NAMES)

  num_val = sum(1 for _ in val_path.rglob('*.jpg')) + sum(1 for _ in val_path.rglob('*.png'))
  print(f"\n Loaded {NUM_CLASSES} classes from dataset")
  print(f"  Validation: {num_val} images")
  print(f"  Meta-kategorii: {len(META_CATEGORIES)}")
  print(f"\n  Pierwsze 10 classes: {CLASS_NAMES[:10]}")
  print(f"  Ostatnie 5 classes: {CLASS_NAMES[-5:]}")
else:
  print("  No val folder - classes will be loaded from checkpoint")

# 2. Load Models

In [ ]:
# Swin model loader

def load_swin_model(config, num_classes):
  print(f"\n Loading: {config['display_name']}")

  checkpoint = torch.load(config['checkpoint'], map_location='cpu', weights_only=False)

  # Check if checkpoint contains class info
  checkpoint_classes = checkpoint.get('classes', None)

  # Liczba classes: z 'classes' > z 'config' > z argumentu
  if checkpoint_classes:
    actual_num_classes = len(checkpoint_classes)
  elif 'config' in checkpoint:
    actual_num_classes = checkpoint['config'].get('num_classes', num_classes)
  else:
    actual_num_classes = num_classes

  model = timm.create_model(config['model_name'], pretrained=False, num_classes=actual_num_classes)

  info = {'model_name': config['model_name'], 'img_size': config['img_size']}

  if isinstance(checkpoint, dict):
    # Handle different checkpoint formats
    # Strategic: ema_state_dict, model_state_dict, best_acc
    # Tactical: ema, model, best
    if 'ema_state_dict' in checkpoint:
      state_dict = checkpoint['ema_state_dict']
    elif 'ema' in checkpoint:
      state_dict = checkpoint['ema']
    elif 'model_state_dict' in checkpoint:
      state_dict = checkpoint['model_state_dict']
    elif 'model' in checkpoint:
      state_dict = checkpoint['model']
    elif 'state_dict' in checkpoint:
      state_dict = checkpoint['state_dict']
    else:
      state_dict = checkpoint

    # Best accuracy - different key names
    info['best_acc'] = checkpoint.get('best_acc', checkpoint.get('best', 'N/A'))
    info['epoch'] = checkpoint.get('epoch', 'N/A')

    # Save classes from checkpoint
    if checkpoint_classes:
      info['classes'] = checkpoint_classes
      print(f"  Classes z checkpointu: {len(checkpoint_classes)}")
  else:
    state_dict = checkpoint

  model.load_state_dict(state_dict, strict=True)
  model = model.to(DEVICE).eval()

  info['num_classes'] = actual_num_classes
  info['num_params'] = sum(p.numel() for p in model.parameters())
  info['num_params_m'] = info['num_params'] / 1e6
  print(f"  Parameters: {info['num_params_m']:.2f}M, Classes: {actual_num_classes}")

  if FVCORE_AVAILABLE:
    try:
      dummy = torch.randn(1, 3, config['img_size'], config['img_size']).to(DEVICE)
      flops = FlopCountAnalysis(model, dummy)
      info['gflops'] = flops.total() / 1e9
      print(f"  GFLOPs: {info['gflops']:.2f}")
    except:
      info['gflops'] = 'N/A'
  else:
    info['gflops'] = 'N/A'

  return model, info

# Load models
models, model_info = {}, {}
for tier, config in SWIN_PATHS.items():
  if config['checkpoint'].exists():
    models[tier], model_info[tier] = load_swin_model(config, NUM_CLASSES)

    # If CLASS_NAMES not from dataset, use from checkpoint
    if CLASS_NAMES is None and 'classes' in model_info[tier]:
      CLASS_NAMES = model_info[tier]['classes']
      NUM_CLASSES = len(CLASS_NAMES)
      META_CATEGORIES = build_meta_categories(CLASS_NAMES)
      print(f"\n Using classes from checkpoint: {NUM_CLASSES}")
  else:
    print(f" Missing: {config['checkpoint']}")

print(f"\n Loaded {len(models)} modeli")
print(f" Finalna liczba classes: {NUM_CLASSES}")
if CLASS_NAMES:
  print(f"  Examples: {CLASS_NAMES[:5]} ... {CLASS_NAMES[-3:]}")

# 3. Dataset & Evaluation

In [ ]:
# Dataset class

class ClassificationDataset(Dataset):
  def __init__(self, root, split, img_size, class_names):
    self.root = root / split
    self.img_size = img_size
    self.class_to_idx = {c: i for i, c in enumerate(class_names)}

    self.samples = []
    for class_dir in self.root.iterdir():
      if class_dir.is_dir() and class_dir.name in self.class_to_idx:
        idx = self.class_to_idx[class_dir.name]
        for ext in ['*.jpg', '*.png']:
          for img in class_dir.glob(ext):
            self.samples.append((img, idx))

    self.transform = A.Compose([
      A.Resize(img_size, img_size),
      A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
      ToTensorV2()
    ])

  def __len__(self): return len(self.samples)

  def __getitem__(self, idx):
    path, label = self.samples[idx]
    img = np.array(Image.open(path).convert('RGB'))
    img = self.transform(image=img)['image']
    return img, label, str(path)

# Create datasets
print(" Creating datasets...")
datasets, dataloaders = {}, {}
for tier, config in SWIN_PATHS.items():
  if tier in models:
    ds = ClassificationDataset(DATASET_PATH, 'val', config['img_size'], CLASS_NAMES)
    datasets[tier] = ds
    dataloaders[tier] = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    print(f"  {tier}: {len(ds)} images")

In [ ]:
# Model evaluation

@torch.no_grad()
def evaluate_model(model, dataloader, class_names):
  model.eval()
  all_preds, all_labels, all_probs, all_paths = [], [], [], []

  for images, labels, paths in tqdm(dataloader, desc="Evaluating"):
    images = images.to(DEVICE)
    outputs = model(images)
    probs = F.softmax(outputs, dim=1)
    preds = outputs.argmax(dim=1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.numpy())
    all_probs.extend(probs.cpu().numpy())
    all_paths.extend(paths)

  all_preds = np.array(all_preds)
  all_labels = np.array(all_labels)
  all_probs = np.array(all_probs)

  top1 = accuracy_score(all_labels, all_preds)
  top5 = top_k_accuracy_score(all_labels, all_probs, k=5, labels=range(len(class_names)))
  cm = confusion_matrix(all_labels, all_preds)
  per_class = cm.diagonal() / (cm.sum(axis=1) + 1e-10)

  return {
    'top1_acc': top1, 'top5_acc': top5,
    'confusion_matrix': cm,
    'per_class_accuracy': per_class,
    'predictions': all_preds, 'labels': all_labels,
    'probabilities': all_probs, 'paths': all_paths
  }

# Ewaluuj
results = {}
for tier in models.keys():
  print(f"\n{'='*50}")
  print(f" Evaluating: {SWIN_PATHS[tier]['display_name']}")
  results[tier] = evaluate_model(models[tier], dataloaders[tier], CLASS_NAMES)
  print(f"  Top-1: {results[tier]['top1_acc']*100:.2f}%")
  print(f"  Top-5: {results[tier]['top5_acc']*100:.2f}%")

# 4. CONFUSION MATRICES

In [ ]:
# Confusion matrix (full 56x56, no annotations)I

def plot_confusion_matrix(cm, class_names, title, save_path):
  """Confusion matrix - tylko heatmapa, bez liczb."""
  fig, ax = plt.subplots(figsize=(24, 20))

  cm_display = cm.astype(float)
  cm_display[cm_display == 0] = np.nan

  sns.heatmap(cm_display, annot=False, cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, cbar_kws={'label': 'Sample count'})

  ax.set_xlabel('Predicted', fontsize=14)
  ax.set_ylabel('True', fontsize=14)
  ax.set_title(title, fontsize=16, fontweight='bold')
  plt.xticks(rotation=90, fontsize=7)
  plt.yticks(rotation=0, fontsize=7)
  plt.tight_layout()
  plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
  plt.show()
  plt.close()
  print(f"  {save_path.name}")

# Top confusions

def plot_top_confusions(cm, class_names, title, save_path, top_n=20):
  confusions = []
  for i in range(len(class_names)):
    for j in range(len(class_names)):
      if i != j and cm[i, j] > 0:
        confusions.append({
          'true': class_names[i],
          'pred': class_names[j],
          'count': cm[i, j],
          'label': f"{class_names[i]} → {class_names[j]}"
        })

  confusions = sorted(confusions, key=lambda x: x['count'], reverse=True)[:top_n]

  if not confusions:
    print("   No confusions")
    return []

  fig, ax = plt.subplots(figsize=(12, 8))
  labels = [c['label'] for c in confusions]
  counts = [c['count'] for c in confusions]

  bars = ax.barh(range(len(confusions)), counts, color='#e74c3c', edgecolor='black')
  ax.set_yticks(range(len(confusions)))
  ax.set_yticklabels(labels, fontsize=9)
  ax.set_xlabel('Confusion count', fontsize=12)
  ax.set_title(title, fontsize=14, fontweight='bold')
  ax.invert_yaxis()

  for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
        f'{int(count)}', va='center', fontsize=9)

  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  plt.close()
  print(f"  {save_path.name}")
  return confusions

# CONFUSION MATRIX - POGRUPOWANA - TYLKO LICZBY

def plot_cm_grouped(cm, class_names, meta_categories, title, save_path):
  class_to_meta = {}
  for meta, classes in meta_categories.items():
    for c in classes:
      if c in class_names:
        class_to_meta[class_names.index(c)] = meta

  meta_names = list(meta_categories.keys())
  n_meta = len(meta_names)
  cm_grouped = np.zeros((n_meta, n_meta))

  for i in range(len(class_names)):
    for j in range(len(class_names)):
      if i in class_to_meta and j in class_to_meta:
        mi = meta_names.index(class_to_meta[i])
        mj = meta_names.index(class_to_meta[j])
        cm_grouped[mi, mj] += cm[i, j]

  fig, ax = plt.subplots(figsize=(14, 12))
  sns.heatmap(cm_grouped, annot=True, fmt='.0f', cmap='Blues',
        xticklabels=meta_names, yticklabels=meta_names, ax=ax,
        cbar_kws={'label': 'Sample count'}, annot_kws={'size': 8})
  ax.set_xlabel('Predicted Meta-Category', fontsize=12)
  ax.set_ylabel('True Meta-Category', fontsize=12)
  ax.set_title(title, fontsize=14, fontweight='bold')
  plt.xticks(rotation=45, ha='right', fontsize=10)
  plt.yticks(rotation=0, fontsize=10)
  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  plt.close()
  print(f"  {save_path.name}")

  row_sums = cm_grouped.sum(axis=1) + 1e-10
  per_meta_acc = np.diag(cm_grouped) / row_sums
  return per_meta_acc.mean()

# Generate

print("\n Confusion Matrices...")

meta_accs = {}
top_confusions = {}

for tier, res in results.items():
  print(f"\n{'='*50}")
  print(f" {SWIN_PATHS[tier]['display_name']}")

  plot_confusion_matrix(
    res['confusion_matrix'], CLASS_NAMES,
    f"Confusion Matrix - {SWIN_PATHS[tier]['display_name']} (Top-1: {res['top1_acc']*100:.2f}%)",
    OUTPUT_DIRS['confusion_matrices'] / f'confusion_matrix_{tier}.png'
  )

  top_confusions[tier] = plot_top_confusions(
    res['confusion_matrix'], CLASS_NAMES,
    f"Top 20 Confusions - {SWIN_PATHS[tier]['display_name']}",
    OUTPUT_DIRS['confusion_matrices'] / f'top_confusions_{tier}.png'
  )

  meta_accs[tier] = plot_cm_grouped(
    res['confusion_matrix'], CLASS_NAMES, META_CATEGORIES,
    f"Meta-Category CM - {SWIN_PATHS[tier]['display_name']}",
    OUTPUT_DIRS['confusion_matrices'] / f'confusion_matrix_{tier}_grouped.png'
  )
  print(f"  Meta-accuracy: {meta_accs[tier]*100:.2f}%")

print("\n" + "="*60)
print(" TOP 10 CONFUSIONS")
for tier, confusions in top_confusions.items():
  if confusions:
    print(f"\n{SWIN_PATHS[tier]['display_name']}:")
    for i, c in enumerate(confusions[:10], 1):
      print(f"  {i:2d}. {c['true']:20s} → {c['pred']:20s} : {c['count']:3d}")

# 5. Training Curves

In [ ]:
# Search for training log files
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Inżynierka")

print(" Searching for training log files...\n")

# Search in checkpoint folders
for pattern in ['**/training*.json', '**/log*.json', '**/history*.json', '**/metrics*.json']:
  found = list(BASE.glob(pattern))
  if found:
    print(f" Pattern: {pattern}")
    for f in found[:10]:
      size = f.stat().st_size / 1024
      print(f"  {f.relative_to(BASE)} ({size:.1f} KB)")
    print()

# Check swin folders specifically
print("\n Swin folder contents:")
swin_dirs = [
  BASE / 'Strategic_components/checkpoints/swin/strategic_swin_base_384',
  BASE / 'Medium_components/checkpoints/swin/medium_swin_small_224',
]

for d in swin_dirs:
  if d.exists():
    print(f"\n{d.name}:")
    for f in d.iterdir():
      size = f.stat().st_size / 1024 / 1024
      print(f"  {f.name} ({size:.2f} MB)")
  else:
    print(f"\n Nie istnieje: {d}")

In [ ]:
# Training curves - reconstructed from checkpoints

def extract_training_history(checkpoint_dir, tier_name):
  """Extract history from epoch_*.pth checkpoints."""
  checkpoint_dir = Path(checkpoint_dir)
  history = []

  # Find checkpoints
  patterns = ['epoch_*.pth', 'ep*.pth']
  checkpoints = []
  for pat in patterns:
    checkpoints.extend(checkpoint_dir.glob(pat))

  # Dodaj best i last
  for name in ['best.pth', 'last.pth']:
    p = checkpoint_dir / name
    if p.exists():
      checkpoints.append(p)

  print(f"  Found {len(checkpoints)} checkpoints")

  for cp_path in sorted(checkpoints):
    try:
      cp = torch.load(cp_path, map_location='cpu', weights_only=False)
      epoch = cp.get('epoch', 0)
      best_acc = cp.get('best_acc', cp.get('best', 0))

      # Some checkpoints may have more info
      train_loss = cp.get('train_loss', None)
      val_loss = cp.get('val_loss', None)
      val_acc = cp.get('val_acc', best_acc)

      history.append({
        'file': cp_path.name,
        'epoch': epoch,
        'best_acc': best_acc,
        'val_acc': val_acc,
        'train_loss': train_loss,
        'val_loss': val_loss
      })
      print(f"   {cp_path.name}: epoch={epoch}, acc={best_acc:.4f}")
    except Exception as e:
      print(f"    {cp_path.name}: {e}")

  return sorted(history, key=lambda x: x['epoch'])

def plot_reconstructed_curves(history, title, save_path):
  """Rysuje krzywe z zrekonstruowanej historii."""
  if not history:
    print("  No data")
    return

  epochs = [h['epoch'] for h in history]
  best_accs = [h['best_acc'] for h in history]

  fig, ax = plt.subplots(figsize=(10, 6))
  ax.plot(epochs, best_accs, 'go-', lw=2, markersize=8, label='Best Accuracy')
  ax.scatter(epochs, best_accs, c='green', s=100, zorder=5)

  # Annotacje
  for i, h in enumerate(history):
    ax.annotate(f"{h['best_acc']:.3f}", (h['epoch'], h['best_acc']),
          textcoords="offset points", xytext=(0, 10), ha='center', fontsize=8)

  ax.set_xlabel('Epoch', fontsize=12)
  ax.set_ylabel('Accuracy', fontsize=12)
  ax.set_title(title, fontsize=14, fontweight='bold')
  ax.grid(alpha=0.3)
  ax.legend()
  ax.set_ylim(min(best_accs) - 0.02, 1.0)

  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  plt.close()
  print(f"  {save_path.name}")

# Generate

print("\n Reconstructing training curves from checkpoints...")

training_history = {}

checkpoint_dirs = {
  'strategic': BASE_PATH / 'Strategic_components/checkpoints/swin/strategic_swin_base_384',
  'tactical': BASE_PATH / 'Medium_components/checkpoints/swin/medium_swin_small_224',
}

for tier, cp_dir in checkpoint_dirs.items():
  print(f"\n{'='*50}")
  print(f" {SWIN_PATHS[tier]['display_name']}")

  if cp_dir.exists():
    history = extract_training_history(cp_dir, tier)
    training_history[tier] = history

    if history:
      plot_reconstructed_curves(
        history,
        f"Training Progress - {SWIN_PATHS[tier]['display_name']}",
        OUTPUT_DIRS['training_curves'] / f'training_{tier}.png'
      )
  else:
    print(f"  Folder does not exist: {cp_dir}")

# Summary
print("\n" + "="*60)
print(" TRAINING SUMMARY")
print("="*60)
for tier, history in training_history.items():
  if history:
    final = history[-1]
    best = max(history, key=lambda x: x['best_acc'])
    print(f"\n{SWIN_PATHS[tier]['display_name']}:")
    print(f"  Ostatni epoch: {final['epoch']}")
    print(f"  Best accuracy: {best['best_acc']:.4f} (epoch {best['epoch']})")

In [ ]:
# Training curves - reconstructed from checkpoints
import re

def extract_training_history(checkpoint_dir):
  """Extract history from checkpoints - epoch from filename."""
  checkpoint_dir = Path(checkpoint_dir)
  history = []

  patterns = ['epoch_*.pth', 'ep*.pth']
  checkpoints = []
  for pat in patterns:
    checkpoints.extend(checkpoint_dir.glob(pat))

  print(f"  Found {len(checkpoints)} checkpoints")

  for cp_path in checkpoints:
    try:
      # Extract epoch from filename (epoch_5.pth -> 5)
      match = re.search(r'(?:epoch_|ep)(\d+)', cp_path.name)
      if match:
        epoch = int(match.group(1))
      else:
        continue

      cp = torch.load(cp_path, map_location='cpu', weights_only=False)
      best_acc = cp.get('best_acc', cp.get('best', 0))

      history.append({
        'file': cp_path.name,
        'epoch': epoch,
        'best_acc': float(best_acc),
      })
      print(f"   {cp_path.name}: epoch={epoch}, acc={best_acc:.4f}")
    except Exception as e:
      print(f"    {cp_path.name}: {e}")

  # Sortuj po epoce
  return sorted(history, key=lambda x: x['epoch'])

def plot_reconstructed_curves(history, title, save_path):
  if not history:
    print("  No data")
    return

  epochs = [h['epoch'] for h in history]
  best_accs = [h['best_acc'] for h in history]

  fig, ax = plt.subplots(figsize=(10, 6))
  ax.plot(epochs, best_accs, 'go-', lw=2, markersize=10, label='Best Accuracy')

  for h in history:
    ax.annotate(f"{h['best_acc']:.3f}", (h['epoch'], h['best_acc']),
          textcoords="offset points", xytext=(0, 12), ha='center', fontsize=9)

  ax.set_xlabel('Epoch', fontsize=12)
  ax.set_ylabel('Accuracy', fontsize=12)
  ax.set_title(title, fontsize=14, fontweight='bold')
  ax.grid(alpha=0.3)
  ax.legend(loc='lower right')

  # X axis - integer epochs only
  ax.set_xticks(epochs)
  ax.set_xticklabels([str(e) for e in epochs])

  # Y axis
  min_acc = min(best_accs)
  ax.set_ylim(max(0, min_acc - 0.05), 1.0)

  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  plt.close()
  print(f"  {save_path.name}")

# Generate

print("\n Training curves from checkpoints...")

training_history = {}

checkpoint_dirs = {
  'strategic': BASE_PATH / 'Strategic_components/checkpoints/swin/strategic_swin_base_384',
  'tactical': BASE_PATH / 'Medium_components/checkpoints/swin/medium_swin_small_224',
}

for tier, cp_dir in checkpoint_dirs.items():
  print(f"\n{'='*50}")
  print(f" {SWIN_PATHS[tier]['display_name']}")

  if cp_dir.exists():
    history = extract_training_history(cp_dir)
    training_history[tier] = history

    if history:
      plot_reconstructed_curves(
        history,
        f"Training Progress - {SWIN_PATHS[tier]['display_name']}",
        OUTPUT_DIRS['training_curves'] / f'training_{tier}.png'
      )
  else:
    print(f"  Brak folderu: {cp_dir}")

print("\n" + "="*60)
print(" TRAINING SUMMARY")
for tier, history in training_history.items():
  if history:
    best = max(history, key=lambda x: x['best_acc'])
    final = history[-1]
    print(f"\n{SWIN_PATHS[tier]['display_name']}:")
    print(f"  Epochs: {history[0]['epoch']} → {final['epoch']}")
    print(f"  Best: {best['best_acc']:.4f} @ epoch {best['epoch']}")

# 6. GradCAM

In [ ]:
# Check confidence distribution for correct predictions
for tier, res in results.items():
  correct_confs = []
  for idx in range(len(res['predictions'])):
    pred, true = res['predictions'][idx], res['labels'][idx]
    if pred == true:
      conf = res['probabilities'][idx][pred]
      correct_confs.append(conf)

  correct_confs = np.array(correct_confs)
  print(f"\n{tier}:")
  print(f"  Min: {correct_confs.min():.4f}")
  print(f"  Max: {correct_confs.max():.4f}")
  print(f"  Mean: {correct_confs.mean():.4f}")
  print(f"  Median: {np.median(correct_confs):.4f}")
  print(f"  >0.9: {(correct_confs > 0.9).sum()}")
  print(f"  >0.8: {(correct_confs > 0.8).sum()}")
  print(f"  >0.7: {(correct_confs > 0.7).sum()}")

In [ ]:
# GradCAM for Swin (corrected)

# Create gradcam folder
GRADCAM_DIR = Path("/content/drive/MyDrive/Inżynierka/Raporty/Swin/gradcam")
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

# Funkcja reshape dla Swin (output [B, H, W, C] → [B, C, H, W])
def swin_reshape_transform(tensor):
  if len(tensor.shape) == 4:
    return tensor.permute(0, 3, 1, 2)
  return tensor

def generate_gradcam(model, img_path, img_size, class_names):
  """Generuje GradCAM dla Swin Transformer."""
  img = Image.open(img_path).convert('RGB')
  img_np = np.array(img.resize((img_size, img_size))) / 255.0

  transform = A.Compose([
    A.Resize(img_size, img_size),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
  ])
  input_tensor = transform(image=np.array(img))['image'].unsqueeze(0).to(DEVICE)

  with torch.no_grad():
    output = model(input_tensor)
    probs = F.softmax(output, dim=1)
    pred = output.argmax(dim=1).item()
    conf = probs[0, pred].item()

  true_name = Path(img_path).parent.name
  true = class_names.index(true_name) if true_name in class_names else -1

  # Correct layer for Swin + reshape_transform
  target_layers = [model.layers[-1].blocks[-1].norm1]
  cam = GradCAM(
    model=model,
    target_layers=target_layers,
    reshape_transform=swin_reshape_transform
  )
  grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred)])[0]
  vis = show_cam_on_image(img_np.astype(np.float32), grayscale_cam, use_rgb=True)

  return vis, pred, conf, true

def save_gradcam_grid(samples, class_names, save_path, title, cols=4):
  """Save GradCAM visualization grid."""
  n = len(samples)
  if n == 0: return
  rows = (n + cols - 1) // cols

  fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4.5*rows))
  axes = np.array(axes).flatten() if n > 1 else [axes]

  for idx, (path, vis, pred, conf, true) in enumerate(samples):
    ax = axes[idx]
    ax.imshow(vis)
    pred_name = class_names[pred] if pred < len(class_names) else "?"
    true_name = class_names[true] if 0 <= true < len(class_names) else "?"
    color = 'green' if pred == true else 'red'
    ax.set_title(f"Pred: {pred_name}\nTrue: {true_name}\nConf: {conf:.1%}", fontsize=8, color=color)
    ax.axis('off')

  for idx in range(n, len(axes)): axes[idx].axis('off')
  plt.suptitle(title, fontsize=12, fontweight='bold')
  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  plt.close()
  print(f"  {save_path.name}")

# Generate GradCAM

NUM_SAMPLES = 8
random.seed(42)

# Adjusted thresholds (model max ~0.89)
CONF_HIGH = 0.8  # Pewne: >80%
CONF_LOW = 0.5   # Niepewne: 50-80%

print("\n GradCAM...")
print(f"  Progi: Pewne >{CONF_HIGH:.0%}, Niepewne {CONF_LOW:.0%}-{CONF_HIGH:.0%}")
gradcam_stats = {}

for tier in models.keys():
  print(f"\n{'='*50}")
  print(f" {SWIN_PATHS[tier]['display_name']}")

  model = models[tier]
  config = SWIN_PATHS[tier]
  res = results[tier]

  confident, uncertain, incorrect = [], [], []
  for idx in range(len(res['predictions'])):
    pred, true = res['predictions'][idx], res['labels'][idx]
    conf = res['probabilities'][idx][pred]
    path = res['paths'][idx]

    if pred == true and conf > CONF_HIGH:
      confident.append((path, pred, conf, true))
    elif pred == true and CONF_LOW <= conf <= CONF_HIGH:
      uncertain.append((path, pred, conf, true))
    elif pred != true:
      incorrect.append((path, pred, conf, true))

  print(f"  Pewne (>{CONF_HIGH:.0%}): {len(confident)}")
  print(f"  Niepewne ({CONF_LOW:.0%}-{CONF_HIGH:.0%}): {len(uncertain)}")
  print(f"  Incorrect: {len(incorrect)}")
  gradcam_stats[tier] = {'confident': len(confident), 'uncertain': len(uncertain), 'incorrect': len(incorrect)}

  categories = {
    'confident': random.sample(confident, min(NUM_SAMPLES, len(confident))),
    'uncertain': random.sample(uncertain, min(NUM_SAMPLES, len(uncertain))),
    'incorrect': random.sample(incorrect, min(NUM_SAMPLES, len(incorrect)))
  }

  for cat_name, samples in categories.items():
    if not samples: continue
    print(f"  Generating {cat_name}...")

    cat_dir = GRADCAM_DIR / tier / cat_name
    cat_dir.mkdir(parents=True, exist_ok=True)

    gradcam_samples = []
    for path, pred, conf, true in tqdm(samples, leave=False):
      try:
        vis, _, _, _ = generate_gradcam(model, path, config['img_size'], CLASS_NAMES)
        gradcam_samples.append((path, vis, pred, conf, true))
        Image.fromarray(vis).save(cat_dir / f"{Path(path).stem}_gradcam.png")
      except Exception as e:
        print(f"  {Path(path).name}: {e}")

    if gradcam_samples:
      save_gradcam_grid(
        gradcam_samples, CLASS_NAMES,
        GRADCAM_DIR / tier / f'{cat_name}_grid.png',
        f"GradCAM - {SWIN_PATHS[tier]['display_name']} - {cat_name.upper()}"
      )

# Summary
print("\n" + "="*60)
print(" STATYSTYKI GRADCAM")
for tier, stats in gradcam_stats.items():
  print(f"\n{SWIN_PATHS[tier]['display_name']}:")
  print(f"  Pewne (>{CONF_HIGH:.0%}): {stats['confident']}")
  print(f"  Niepewne ({CONF_LOW:.0%}-{CONF_HIGH:.0%}): {stats['uncertain']}")
  print(f"  Incorrect: {stats['incorrect']}")

# 7. Tables & Comparisons

In [ ]:
# Tables

print("\n Generating tables...")

# Summary table
metrics_data = []
for tier, res in results.items():
  config = SWIN_PATHS[tier]
  info = model_info.get(tier, {})
  metrics_data.append({
    'Tier': config['tier_name'],
    'Model': config['display_name'],
    'img_size': config['img_size'],
    'Params (M)': f"{info.get('num_params_m', 0):.1f}",
    'GFLOPs': f"{info.get('gflops', 'N/A'):.2f}" if isinstance(info.get('gflops'), (int, float)) else 'N/A',
    'Top-1 (%)': f"{res['top1_acc']*100:.2f}",
    'Top-5 (%)': f"{res['top5_acc']*100:.2f}",
  })

df_metrics = pd.DataFrame(metrics_data)
print("\n" + df_metrics.to_string(index=False))
df_metrics.to_csv(OUTPUT_DIRS['tables'] / 'metrics_summary.csv', index=False)

# LaTeX
latex = """\\begin{table}[htbp]
\\centering
\\caption{Classification model comparison Swin Transformer}
\\label{tab:swin_metrics}
\\begin{tabular}{llccccc}
\\toprule
Tier & Model & img\\_size & Params & GFLOPs & Top-1 & Top-5 \\\\
\\midrule
"""
for _, r in df_metrics.iterrows():
  latex += f"{r['Tier']} & {r['Model']} & {r['img_size']} & {r['Params (M)']} & {r['GFLOPs']} & {r['Top-1 (%)']} & {r['Top-5 (%)']} \\\\\n"
latex += """\\bottomrule
\\end{tabular}
\\end{table}
"""

with open(OUTPUT_DIRS['tables'] / 'metrics_summary.tex', 'w') as f:
  f.write(latex)
print("  metrics_summary.csv, .tex")

# Per-class accuracy
per_class_data = []
for i, cn in enumerate(CLASS_NAMES):
  row = {'Class': cn}
  for tier, res in results.items():
    row[f"{SWIN_PATHS[tier]['tier_name']}_Acc"] = f"{res['per_class_accuracy'][i]*100:.2f}"
  per_class_data.append(row)

df_per_class = pd.DataFrame(per_class_data)
df_per_class.to_csv(OUTPUT_DIRS['tables'] / 'per_class_accuracy.csv', index=False)
print("  per_class_accuracy.csv")

In [ ]:
# Comparison plots

if len(results) >= 2:
  print("\n Generating comparison plots...")

  tiers = list(results.keys())
  labels = [SWIN_PATHS[t]['display_name'] for t in tiers]
  top1 = [results[t]['top1_acc']*100 for t in tiers]
  top5 = [results[t]['top5_acc']*100 for t in tiers]
  colors = ['#2ecc71', '#3498db']

  fig, axes = plt.subplots(1, 2, figsize=(12, 5))

  bars1 = axes[0].bar(labels, top1, color=colors, edgecolor='black')
  axes[0].set_ylabel('Top-1 Accuracy (%)')
  axes[0].set_title('Top-1 Accuracy', fontweight='bold')
  axes[0].set_ylim(0, 100)
  axes[0].bar_label(bars1, fmt='%.2f%%')
  axes[0].grid(alpha=0.3, axis='y')

  bars2 = axes[1].bar(labels, top5, color=colors, edgecolor='black')
  axes[1].set_ylabel('Top-5 Accuracy (%)')
  axes[1].set_title('Top-5 Accuracy', fontweight='bold')
  axes[1].set_ylim(0, 100)
  axes[1].bar_label(bars2, fmt='%.2f%%')
  axes[1].grid(alpha=0.3, axis='y')

  plt.tight_layout()
  plt.savefig(OUTPUT_DIRS['comparisons'] / 'accuracy_comparison.png', dpi=150, facecolor='white')
  plt.show()
  plt.close()
  print("  accuracy_comparison.png")

  # Per-class comparison
  if 'strategic' in results and 'tactical' in results:
    fig, ax = plt.subplots(figsize=(16, 8))
    x = np.arange(len(CLASS_NAMES))
    w = 0.35

    ax.bar(x - w/2, results['strategic']['per_class_accuracy']*100, w, label='Strategic', color='#2ecc71', alpha=0.8)
    ax.bar(x + w/2, results['tactical']['per_class_accuracy']*100, w, label='Tactical', color='#3498db', alpha=0.8)

    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Per-Class Accuracy Comparison', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=6)
    ax.legend()
    ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIRS['comparisons'] / 'per_class_comparison.png', dpi=150, facecolor='white')
    plt.show()
    plt.close()
    print("  per_class_comparison.png")

# 8. Final report

In [ ]:
# Generate markdown report

print("\n Generating report...")

# Progi (te same co w GradCAM)
CONF_HIGH = 0.8
CONF_LOW = 0.5

report = f"""# REPORT 6.2 - SWIN TRANSFORMER CLASSIFIER EVALUATION

Data: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}
Liczba classes: {NUM_CLASSES}

---

## 1. METRICS SUMMARY

| Tier | Model | img_size | Params | Top-1 Acc | Top-5 Acc |
|------|-------|----------|--------|-----------|------------|
"""

for tier, res in results.items():
  config = SWIN_PATHS[tier]
  info = model_info.get(tier, {})
  report += f"| {config['tier_name']} | {config['display_name']} | {config['img_size']} | {info.get('num_params_m', 0):.1f}M | **{res['top1_acc']*100:.2f}%** | {res['top5_acc']*100:.2f}% |\n"

report += f"""
---

## 2. DETAILED RESULTS

"""

for tier, res in results.items():
  config = SWIN_PATHS[tier]
  per_class = res['per_class_accuracy']
  best_5 = np.argsort(per_class)[-5:][::-1]
  worst_5 = np.argsort(per_class)[:5]

  report += f"""### {config['display_name']}

- **Top-1 Accuracy**: {res['top1_acc']*100:.2f}%
- **Top-5 Accuracy**: {res['top5_acc']*100:.2f}%
- **Meta-Category Accuracy**: {meta_accs.get(tier, 0)*100:.2f}%

**Najlepsze classesy:**
"""
  for idx in best_5:
    report += f"- {CLASS_NAMES[idx]}: {per_class[idx]*100:.2f}%\n"

  report += "\n**Najgorsze classesy:**\n"
  for idx in worst_5:
    report += f"- {CLASS_NAMES[idx]}: {per_class[idx]*100:.2f}%\n"

  stats = gradcam_stats.get(tier, {})
  report += f"""
**Statystyki predykcji:**
- Pewne (>{CONF_HIGH:.0%} conf): {stats.get('confident', 0)}
- Niepewne ({CONF_LOW:.0%}-{CONF_HIGH:.0%}): {stats.get('uncertain', 0)}
- Incorrect: {stats.get('incorrect', 0)}

"""

report += f"""---

## 3. GENERATED FILES

### Tabele:
- `tables/metrics_summary.csv` - summary metrics
- `tables/metrics_summary.tex` - LaTeX table
- `tables/per_class_accuracy.csv` - accuracy per classesa

### Confusion Matrices:
- `confusion_matrices/confusion_matrix_strategic.png` - full {NUM_CLASSES}x{NUM_CLASSES}
- `confusion_matrices/confusion_matrix_tactical.png` - full {NUM_CLASSES}x{NUM_CLASSES}
- `confusion_matrices/confusion_matrix_*_grouped.png` - pogrupowane meta-kategorie
- `confusion_matrices/top_confusions_*.png` - top 20 top confusions

### Training curves:
- `training_curves/training_strategic.png`
- `training_curves/training_tactical.png`

### GradCAM:
- `gradcam/strategic/confident/` - pewne predykcje (>{CONF_HIGH:.0%})
- `gradcam/strategic/uncertain/` - niepewne predykcje ({CONF_LOW:.0%}-{CONF_HIGH:.0%})
- `gradcam/strategic/incorrect/` - incorrect predictions
- `gradcam/tactical/...` - analogicznie
- `gradcam/*/confident_grid.png` - siatki wizualizacji

### Comparisons:
- `comparisons/accuracy_comparison.png`
- `comparisons/per_class_comparison.png`

---

## 4. WNIOSKI

"""

if len(results) >= 2:
  diff = results['strategic']['top1_acc'] - results['tactical']['top1_acc']
  report += f"""1. **Tier comparison**: Strategic ({model_info['strategic']['model_name']}) achieves by {abs(diff)*100:.2f}pp {"higher" if diff > 0 else "lower"} Top-1 accuracy than Tactical ({model_info['tactical']['model_name']}).
2. **Cost vs accuracy**: Swin-Base has ~{model_info['strategic']['num_params_m']/model_info['tactical']['num_params_m']:.1f}x more parameters than Swin-Small.
3. **Top-5 accuracy**: Both models achieve very high Top-5 accuracy (~{results['strategic']['top5_acc']*100:.1f}%), which means the correct class is almost always in the top 5 predictions.
4. **Confidence**: Model with {NUM_CLASSES} classes achieves max confidence ~89% (lower than with fewer classes).
"""
else:
  tier = list(results.keys())[0]
  res = results[tier]
  report += f"""1. **Accuracy**: Model achieves {res['top1_acc']*100:.2f}% Top-1 i {res['top5_acc']*100:.2f}% Top-5.
2. **Confidence**: Model with {NUM_CLASSES} classes achieves max confidence ~89%.
"""

report += """
---

*Raport wygenerowany automatycznie przez VSHORAD_6_2_Swin_Evaluation.ipynb*
"""

# Save report
report_path = REPORT_DIR / 'REPORT_6_2_SWIN.md'
with open(report_path, 'w', encoding='utf-8') as f:
  f.write(report)

print(f" Raport: {report_path}")

# Copy plots to report dir

print("\n Copying plots...")

plots_dir = REPORT_DIR / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)

plots_to_copy = [
  (OUTPUT_DIRS['confusion_matrices'] / 'confusion_matrix_strategic.png', 'cm_strategic.png'),
  (OUTPUT_DIRS['confusion_matrices'] / 'confusion_matrix_tactical.png', 'cm_tactical.png'),
  (OUTPUT_DIRS['confusion_matrices'] / 'confusion_matrix_strategic_grouped.png', 'cm_strategic_grouped.png'),
  (OUTPUT_DIRS['confusion_matrices'] / 'confusion_matrix_tactical_grouped.png', 'cm_tactical_grouped.png'),
  (OUTPUT_DIRS['confusion_matrices'] / 'top_confusions_strategic.png', 'top_confusions_strategic.png'),
  (OUTPUT_DIRS['confusion_matrices'] / 'top_confusions_tactical.png', 'top_confusions_tactical.png'),
  (OUTPUT_DIRS['training_curves'] / 'training_strategic.png', 'training_strategic.png'),
  (OUTPUT_DIRS['training_curves'] / 'training_tactical.png', 'training_tactical.png'),
  (OUTPUT_DIRS['comparisons'] / 'accuracy_comparison.png', 'accuracy_comparison.png'),
  (OUTPUT_DIRS['comparisons'] / 'per_class_comparison.png', 'per_class_comparison.png'),
  (GRADCAM_DIR / 'strategic' / 'confident_grid.png', 'gradcam_strategic_confident.png'),
  (GRADCAM_DIR / 'strategic' / 'incorrect_grid.png', 'gradcam_strategic_incorrect.png'),
  (GRADCAM_DIR / 'tactical' / 'confident_grid.png', 'gradcam_tactical_confident.png'),
  (GRADCAM_DIR / 'tactical' / 'incorrect_grid.png', 'gradcam_tactical_incorrect.png'),
]

copied = 0
for src, dst_name in plots_to_copy:
  if src.exists():
    shutil.copy2(src, plots_dir / dst_name)
    print(f"  {dst_name}")
    copied += 1
  else:
    print(f"  Missing: {src.name}")

print(f"\n Copied {copied}/{len(plots_to_copy)} plots")

In [ ]:
# Create ZIP archive

print("\n Tworzenie ZIP...")

zip_path = OUTPUT_DIRS['swin'] / 'REPORT_6_2_SWIN.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
  for root, dirs, files in os.walk(REPORT_DIR):
    for file in files:
      file_path = Path(root) / file
      arcname = file_path.relative_to(REPORT_DIR)
      zf.write(file_path, arcname)

print(f" ZIP: {zip_path}")

In [ ]:
# Summary

print("\n" + "="*60)
print(" REPORT 6.2 GOTOWY!")
print("="*60)

print(f"\n Raport: {REPORT_DIR / 'REPORT_6_2_SWIN.md'}")
print(f" Plots: {REPORT_DIR / 'plots'}")
print(f" ZIP: {zip_path}")

print("\n Results:")
for tier, res in results.items():
  print(f"  {SWIN_PATHS[tier]['display_name']}: Top-1={res['top1_acc']*100:.2f}%, Top-5={res['top5_acc']*100:.2f}%")

print("\n Next steps:")
print("  1. Pobierz REPORT_6_2_SWIN.md")
print("  2. Skopiuj tabele LaTeX do Overleaf")
print("  3. Insert plots into thesis")